<a href="https://colab.research.google.com/github/tomo-mar/project_research_A/blob/main/Highlight_FewShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1, モデルのロード & Driveマウント

In [ ]:
# 1. Driveマウントとディレクトリ設定
from google.colab import drive
import os
import glob

drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/VAD_Experiment"
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. ライブラリとモデルの準備
!pip install -q pydub librosa pandas numpy
!pip install -q git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview
!pip install -q qwen-omni-utils

import torch
import librosa
import re
import numpy as np
import pandas as pd
from pydub import AudioSegment
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor

import logging
logging.getLogger().setLevel(logging.ERROR)

model_path = "Qwen/Qwen2.5-Omni-7B"
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_path, torch_dtype=torch.bfloat16, device_map="auto", attn_implementation="sdpa",
)
processor = Qwen2_5OmniProcessor.from_pretrained(model_path)
print("\n初期設定とモデルロード完了")

2. 学習データ抽出関数の定義（）

In [ ]:
def prepare_examples(input_dir, data_type, max_chunk, train_ids):
    """
    data_type: 1=Full, 2=Uniform, 3=Pairs
    max_chunk: 36以下の偶数
    """
    if max_chunk > 36 or max_chunk % 2 != 0:
        raise ValueError("Max_chunkは36以下の偶数でなくてはならない")

    print(f"データ抽出 (Type: {data_type}, Max_chunk: {max_chunk})")

    # Fullの場合は先頭の1ファイルのみに制限
    if data_type == 1:
        train_ids = train_ids[:1]
        print(f"  -> Fullモード: {train_ids[0]} の全チャンクを使用")

    all_chunks = []

    # 共通: 利用可能な全チャンクをプールに収集
    for tid in train_ids:
        csv_path = os.path.join(input_dir, f"{tid}_label.csv")
        wav_path = os.path.join(input_dir, f"{tid}.wav")
        if not (os.path.exists(csv_path) and os.path.exists(wav_path)):
            continue

        df = pd.read_csv(csv_path)
        audio = AudioSegment.from_wav(wav_path)

        rc = []
        for start_ms in range(0, len(audio), 5000):
            end_ms = min(start_ms + 5000, len(audio))
            chunk = audio[start_ms:end_ms]
            if len(chunk) < 1000: continue
            time_sec = start_ms / 1000.0

            row = df[np.isclose(df['Relative_Time_sec'], time_sec, atol=0.1)]
            if row.empty: continue

            rc.append({
                "rec_id": tid, "time_sec": time_sec,
                "score": float(row['Label_norm'].iloc[0]), "chunk": chunk
            })
        all_chunks.extend(rc)

    if not all_chunks:
        raise ValueError("抽出できるチャンクがない")

    selected_info = []

    # 1. Full (先頭から順にMax_chunk個取得)
    if data_type == 1:
        selected_info = all_chunks[:max_chunk]


    # 2. Uniform (正解ラベルが k/Max_chunk に最も近いものを重複なしで取得)
    elif data_type == 2:
        targets = [(k / max_chunk) for k in range(1, max_chunk + 1)]
        available_indices = set(range(len(all_chunks)))

        for t in targets:
            best_idx, min_diff = -1, float('inf')
            for i in available_indices:
                diff = abs(all_chunks[i]["score"] - t)
                if diff < min_diff:
                    min_diff = diff
                    best_idx = i
            selected_info.append(all_chunks[best_idx])
            available_indices.remove(best_idx)
        selected_info.sort(key=lambda x: x["score"])


    # 3. Pairs (変化率 最大→最小→2番目最大...の交互取得)
    elif data_type == 3:
        pairs = []
        for tid in train_ids:
            rc = sorted([c for c in all_chunks if c["rec_id"] == tid], key=lambda x: x["time_sec"])
            for i in range(len(rc) - 1):
                c1, c2 = rc[i], rc[i+1]
                if abs((c2["time_sec"] - c1["time_sec"]) - 5.0) < 0.1:
                    pairs.append({"c1": c1, "c2": c2, "diff": abs(c2["score"] - c1["score"])})

        pairs.sort(key=lambda x: x["diff"], reverse=True)
        num_pairs = max_chunk // 2

        left, right = 0, len(pairs) - 1
        selected_pairs = []

        while len(selected_pairs) < num_pairs and left <= right:
            selected_pairs.append(pairs[left])
            if len(selected_pairs) == num_pairs: break
            if left != right:
                selected_pairs.append(pairs[right])
            left += 1
            right -= 1

        for p in selected_pairs:
            selected_info.extend([p["c1"], p["c2"]])

    # WAVファイルとして書き出し、リスト化
    examples = []
    for i, info in enumerate(selected_info):
        save_path = f"/content/shot_train_{i}.wav"
        info["chunk"].export(save_path, format="wav")
        examples.append({"path": save_path, "score": round(info["score"], 2)})

    print(f"{len(examples)} 個のチャンクを抽出完了")
    return examples

3. プロンプトと推論関数の定義

In [ ]:
# train=rec05用のプロンプト
# 【重要：スコアリングの制約】
#あなたは些細な音声の変化に対しても過敏に高いスコアを出してしまう傾向があります。以下の基準を厳守し、評価にメリハリをつけてください。
#- スコア0.8以上は、「切り抜き動画のハイライトとして使われるレベルの、明確な感情の爆発（大爆笑、大きな歓声、悲鳴など）」の瞬間にのみ限定してください。
#- 少しの音量変化、通常の会話での軽い笑い、ゲームのSEのみによる音量増加などは、 0.1〜0.3 の低いスコアに抑えてください。
#- スコア0.8以上をつけるシーンは、全体の1割程度の本当に特別な瞬間だけです。

PROMPT_DIRECT = """
あなたは優秀な動画解析アシスタントです。
入力された5秒間の音声クリップを聞き、配信のハイライト（盛り上がり）確率を判定してください。

【評価基準】
0.0に近いほどベースライン（通常のトーン、特筆すべき起伏なし）
1.0に近いほどピーク（絶叫、爆笑など、最高潮の瞬間）

【出力形式】
判定結果となる「0.0 から 1.0 の間の数値（小数点以下2桁）」のみを出力してください。
"""

def run_qwen_inference(target_filepath, few_shot_examples, memory_history, memory_type):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are a highly capable audio analysis AI."}]}
    ]
    audio_arrays = []

    # 学習データの追加
    for shot in few_shot_examples:
        messages.append({
            "role": "user", "content": [
                {"type": "audio", "audio_url": shot["path"]},
                {"type": "text", "text": PROMPT_DIRECT}
            ]
        })
        messages.append({"role": "assistant", "content": [{"type": "text", "text": str(shot["score"])}]})
        arr, _ = librosa.load(shot["path"], sr=16000)
        audio_arrays.append(arr)

    # 記憶の追加
    history_to_use = []
    if memory_type == 1:
        history_to_use = memory_history[-1:] # 直前1つ
    elif memory_type == 2:
        history_to_use = memory_history      # 手前すべて

    for past in history_to_use:
        messages.append({
            "role": "user", "content": [
                {"type": "audio", "audio_url": past["path"]},
                {"type": "text", "text": "【過去の文脈】\nこの5秒間のハイライト確率を教えてください。"}
            ]
        })
        messages.append({"role": "assistant", "content": [{"type": "text", "text": str(past["score"])}]})
        arr, _ = librosa.load(past["path"], sr=16000)
        audio_arrays.append(arr)

    # ターゲットの追加
    if memory_type > 0 and len(history_to_use) > 0:
        target_prompt = "【現在の5秒間】\nこれまでの音声の流れと直前からの変化を考慮して、現在のハイライト確率を 0.0〜1.0 の数値のみで出力してください。"
    else:
        target_prompt = PROMPT_DIRECT

    messages.append({
        "role": "user", "content": [
            {"type": "audio", "audio_url": target_filepath},
            {"type": "text", "text": target_prompt}
        ]
    })
    target_arr, _ = librosa.load(target_filepath, sr=16000)
    audio_arrays.append(target_arr)

    # 推論実行
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=text, audio=audio_arrays, return_tensors="pt", padding=True).to("cuda")

    generate_ids = model.generate(**inputs, max_new_tokens=20, return_audio=False)
    generate_ids = [cur_ids[len(inputs.input_ids[0]):] for cur_ids in generate_ids]
    output_text = processor.batch_decode(generate_ids, skip_special_tokens=True)[0]

    return output_text

def extract_probability(text):
    matches = re.findall(r"0\.\d+|1\.0+", text)
    if matches: return float(matches[-1])
    return 0.0

4. メイン処理

In [ ]:
# 実験パラメータ
# =========================================
DATA_TYPE = 1    # 1: Full, 2: Uniform, 3: Pairs
MEMORY_TYPE = 0  # 0: 無記憶, 1: 直前1つ記憶, 2: 手前すべて記憶
MAX_CHUNK = 36   # 36以下の偶数

# フォルダパスの再設定（trainとtargetを分離）
INPUT_TRAIN_DIR = os.path.join(BASE_DIR, "input", "train")
INPUT_TARGET_DIR = os.path.join(BASE_DIR, "input", "target")
os.makedirs(INPUT_TRAIN_DIR, exist_ok=True)
os.makedirs(INPUT_TARGET_DIR, exist_ok=True)

# フォルダ内の .wav ファイルをIDリスト化
TRAIN_IDS = sorted([f.replace(".wav", "") for f in os.listdir(INPUT_TRAIN_DIR) if f.endswith(".wav")])
TARGET_IDS = sorted([f.replace(".wav", "") for f in os.listdir(INPUT_TARGET_DIR) if f.endswith(".wav")])
# ==========================================

print(f"学習データ({len(TRAIN_IDS)}件): {TRAIN_IDS}")
print(f"推論対象({len(TARGET_IDS)}件): {TARGET_IDS}\n")

if not TRAIN_IDS:
    print(f"エラー: {INPUT_TRAIN_DIR} に学習用wavファイルがない")
elif not TARGET_IDS:
    print(f"エラー: {INPUT_TARGET_DIR} に推論対象のwavファイルがない")
else:
    # 1. データ抽出の実行
    few_shot_examples = prepare_examples(INPUT_TRAIN_DIR, data_type=DATA_TYPE, max_chunk=MAX_CHUNK, train_ids=TRAIN_IDS)

    # 2. 推論対象を順番に処理
    for TARGET_ID in TARGET_IDS:
        target_wav = os.path.join(INPUT_TARGET_DIR, f"{TARGET_ID}.wav")

        print(f"{TARGET_ID} の推論を開始 (Memory:{MEMORY_TYPE})")

        audio = AudioSegment.from_wav(target_wav)
        results = []
        memory_history = []

        chunk_length_ms = 5000
        total_chunks = sum(1 for s in range(0, len(audio), chunk_length_ms) if len(audio[s:min(s + chunk_length_ms, len(audio))]) >= 1000)
        current_chunk = 0

        for start_ms in range(0, len(audio), chunk_length_ms):
            end_ms = min(start_ms + chunk_length_ms, len(audio))
            chunk = audio[start_ms:end_ms]

            if len(chunk) < 1000:
                continue

            current_chunk += 1
            time_sec = start_ms / 1000.0
            print(f"  [{current_chunk}/{total_chunks}]")

            # 現在のチャンクを保存
            temp_chunk_path = f"/content/target_chunk_{current_chunk}.wav"
            chunk.export(temp_chunk_path, format="wav")

            # 推論
            res_text = run_qwen_inference(
                target_filepath=temp_chunk_path,
                few_shot_examples=few_shot_examples,
                memory_history=memory_history,
                memory_type=MEMORY_TYPE
            )
            score = extract_probability(res_text)

            results.append({
                "Time_sec": time_sec,
                "Score_Direct_FewShot": score,
                "RawText": res_text
            })

            # 記憶リストに追加
            memory_history.append({"path": temp_chunk_path, "score": score})

        # csvファイル保存
        type_names = {1: "Full", 2: "Uniform", 3: "Pairs"}
        mem_names = {0: "Mem0", 1: "Mem1", 2: "MemAll"}
        out_filename = f"{TARGET_ID}_{type_names[DATA_TYPE]}_{MAX_CHUNK}chunk_{mem_names[MEMORY_TYPE]}.csv"

        df = pd.DataFrame(results)
        output_csv_path = os.path.join(OUTPUT_DIR, out_filename)
        df.to_csv(output_csv_path, index=False)
        print(f"{TARGET_ID} 完了\n結果を {out_filename} に保存")

    print("\n全ターゲット解析完了")